# 🌿 Realm III: The Leaf Restoration Ritual
### Code Sangam 2026 // Logical Rhythm
**Task:** Blind 4× Super-Resolution — 32×32 → 128×128 leaf images  
**Metric:** Mean Absolute Error (MAE) on [0, 255] uint8 pixel values  

**Strategy:**
- Bilateral pre-denoising of LR inputs (removes thermal sensor noise)
- CBAM Residual Generator with bilinear anchor (starts at ~17.49 MAE baseline)
- Charbonnier loss + frequency-domain auxiliary loss
- Cosine Annealing with Warm Restarts
- 8× Test-Time Augmentation (TTA) at inference

## Cell 1 — Imports & Config

In [ ]:
import os, sys, math, time, random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Environment & Config (Auto-detects PC vs Kaggle) ───────────────────────
IS_KAGGLE     = os.path.exists('/kaggle')
OUTPUT_DIR    = '/kaggle/working' if IS_KAGGLE else '.'
NUM_WORKERS   = 2 if IS_KAGGLE else 0   # 0 on Windows PC to prevent multiprocessing freeze

SEED          = 42
EPOCHS        = 100
BATCH_SIZE    = 16
LR            = 2e-4
NUM_BLOCKS    = 8
NOISE_STD     = 8.0    # Gaussian noise injected during training
FREQ_WEIGHT   = 0.05   # Weight for frequency-domain auxiliary loss
VAL_SPLIT     = 0.1
# ────────────────────────────────────────────────────────────────────────────

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Platform : {"Kaggle Cloud" if IS_KAGGLE else "Local PC"}')
print(f'Device   : {device}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Cell 2 — Dataset Paths

In [ ]:
import glob, os, zipfile

# Auto-detect data directory on both Kaggle and local PC
POSSIBLE_DIRS = [
    '/kaggle/input/competitions/the-leaf-restoration-ritual',
    '/kaggle/input/the-leaf-restoration-ritual',
    'dataset',
    './dataset',
    '.'
]

DATA_DIR = None
for d in POSSIBLE_DIRS:
    if os.path.exists(os.path.join(d, 'train_Low_Resolution')):
        DATA_DIR = d
        break

# Auto-extract zip if dataset folder is not yet extracted on PC
if DATA_DIR is None:
    for zname in ['the-leaf-restoration-ritual.zip', 'dataset.zip']:
        if os.path.exists(zname):
            print(f'Extracting {zname} to dataset/ ...')
            os.makedirs('dataset', exist_ok=True)
            with zipfile.ZipFile(zname, 'r') as z:
                z.extractall('dataset')
            DATA_DIR = 'dataset'
            break

# Auto-detect Kaggle input directory
if DATA_DIR is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train_Low_Resolution' in dirs:
            DATA_DIR = root
            break

assert DATA_DIR is not None, 'Dataset not found! Ensure dataset/ or the-leaf-restoration-ritual.zip exists.'
print(f'\n✅ DATA_DIR = {DATA_DIR}')

LR_TRAIN_DIR  = os.path.join(DATA_DIR, 'train_Low_Resolution')
HR_TRAIN_DIR  = os.path.join(DATA_DIR, 'train_High_Resolution')
LR_TEST_DIR   = os.path.join(DATA_DIR, 'test_Low_Resolution')
TEST_CSV_PATH = os.path.join(DATA_DIR, 'test.csv')
VGG_PATH      = os.path.join(DATA_DIR, 'vgg19_weights.pth')

print(f'Train LR: {len(os.listdir(LR_TRAIN_DIR))} images')
print(f'Train HR: {len(os.listdir(HR_TRAIN_DIR))} images')
print(f'Test  LR: {len(os.listdir(LR_TEST_DIR))} images')


## Cell 3 — Dataset & Dataloaders

In [ ]:
class PairedLeafDataset(Dataset):
    """
    Paired dataset with bilateral denoising + noise-injection augmentation.
    - LR input is bilaterally pre-denoised (removes thermal sensor noise)
    - During training: Gaussian noise re-injected for denoising generalization
    - All geometric augmentations are perfectly synchronized between LR and HR
    """
    def __init__(self, lr_dir, hr_dir=None, filenames=None, is_train=True,
                 bilateral_denoise=True, noise_injection_std=8.0):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.is_train = is_train
        self.bilateral_denoise = bilateral_denoise
        self.noise_injection_std = noise_injection_std
        self.filenames = filenames if filenames is not None else sorted(os.listdir(lr_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        lr_cv = cv2.imread(os.path.join(self.lr_dir, fname))
        lr_cv = cv2.cvtColor(lr_cv, cv2.COLOR_BGR2RGB)

        if self.bilateral_denoise:
            lr_cv = cv2.bilateralFilter(lr_cv, d=7, sigmaColor=30, sigmaSpace=30)

        if self.is_train and self.noise_injection_std > 0:
            noise = np.random.normal(0, self.noise_injection_std, lr_cv.shape).astype(np.float32)
            lr_cv = np.clip(lr_cv.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        lr_t = torch.from_numpy(lr_cv.astype(np.float32) / 255.0).permute(2, 0, 1)

        if self.hr_dir is not None:
            hr_np = np.asarray(Image.open(os.path.join(self.hr_dir, fname)).convert('RGB'), dtype=np.float32) / 255.0
            hr_t = torch.from_numpy(hr_np).permute(2, 0, 1)

            if self.is_train:
                if random.random() > 0.5:
                    lr_t, hr_t = torch.flip(lr_t, [2]), torch.flip(hr_t, [2])
                if random.random() > 0.5:
                    lr_t, hr_t = torch.flip(lr_t, [1]), torch.flip(hr_t, [1])
                k = random.choice([0, 1, 2, 3])
                if k > 0:
                    lr_t = torch.rot90(lr_t, k=k, dims=[1, 2])
                    hr_t = torch.rot90(hr_t, k=k, dims=[1, 2])
            return lr_t, hr_t, fname
        return lr_t, fname


# Train / val split
all_files = sorted(os.listdir(LR_TRAIN_DIR))
rng = np.random.RandomState(SEED)
idxs = np.arange(len(all_files))
rng.shuffle(idxs)
sp = int(len(idxs) * (1.0 - VAL_SPLIT))
train_files = [all_files[i] for i in idxs[:sp]]
val_files   = [all_files[i] for i in idxs[sp:]]

train_ds = PairedLeafDataset(LR_TRAIN_DIR, HR_TRAIN_DIR, train_files, is_train=True,
                              bilateral_denoise=True, noise_injection_std=NOISE_STD)
val_ds   = PairedLeafDataset(LR_TRAIN_DIR, HR_TRAIN_DIR, val_files, is_train=False,
                              bilateral_denoise=True, noise_injection_std=0.0)
test_ds  = PairedLeafDataset(LR_TEST_DIR, hr_dir=None, is_train=False,
                              bilateral_denoise=True, noise_injection_std=0.0)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Bilateral Denoising: ON | Noise Injection Std: {NOISE_STD}')

## Cell 4 — Model Architecture (CBAM Residual Generator)

In [40]:
def init_weights(m):
    """Kaiming Normal init — satisfies Competition Constraint 1 (no pretrained weights)."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None: nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.PReLU(),
            nn.Linear(mid, channels, bias=False)
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.fc(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mx  = self.fc(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        return x * torch.sigmoid(avg + mx)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAMResidualBlock(nn.Module):
    def __init__(self, channels=64, res_scale=0.2):
        super().__init__()
        self.res_scale = res_scale
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.act   = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.ca    = ChannelAttention(channels)
        self.sa    = SpatialAttention()
    def forward(self, x):
        res = self.sa(self.ca(self.conv2(self.act(self.conv1(x)))))
        return x + res * self.res_scale


class LeafSuperResNet(nn.Module):
    """
    4× Super-Resolution Generator.
    INPUT: Bilaterally-denoised 32×32 LR image
    OUTPUT: 128×128 HR image = bilinear_anchor + learned_residual
    """
    def __init__(self, num_blocks=8, num_features=64):
        super().__init__()
        self.shallow   = nn.Conv2d(3, num_features, 3, padding=1)
        self.body      = nn.Sequential(*[CBAMResidualBlock(num_features) for _ in range(num_blocks)])
        self.conv_trunk = nn.Conv2d(num_features, num_features, 3, padding=1)
        self.up1 = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, 3, padding=1),
            nn.PixelShuffle(2), nn.PReLU()
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, 3, padding=1),
            nn.PixelShuffle(2), nn.PReLU()
        )
        self.reconstruct = nn.Conv2d(num_features, 3, 3, padding=1)

    def forward(self, x):
        # Bilinear anchor of bilateral-denoised LR ≈ 17.49 MAE baseline
        anchor = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        feat = self.shallow(x)
        res  = self.conv_trunk(self.body(feat)) + feat
        up   = self.up2(self.up1(res))
        return torch.clamp(anchor + self.reconstruct(up), 0.0, 1.0)


generator = LeafSuperResNet(num_blocks=NUM_BLOCKS).to(device)
generator.apply(init_weights)
n_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
print(f'Generator params: {n_params:,} ({n_params/1e6:.2f}M)')
print('Initialized randomly — Constraint 1 satisfied ✓')

Generator params: 934,693 (0.93M)
Initialized randomly — Constraint 1 satisfied ✓


## Cell 5 — Loss Functions & Metrics

In [41]:
class CharbonnierLoss(nn.Module):
    """Smooth L1 / MAE proxy: sqrt((pred - target)^2 + eps^2)"""
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps2))


class FrequencyLoss(nn.Module):
    """FFT magnitude spectrum L1 — encourages recovery of high-frequency detail."""
    def forward(self, pred, target):
        # Grayscale
        pg = 0.299 * pred[:,0:1]   + 0.587 * pred[:,1:2]   + 0.114 * pred[:,2:3]
        tg = 0.299 * target[:,0:1] + 0.587 * target[:,1:2] + 0.114 * target[:,2:3]
        fp = torch.fft.rfft2(pg, norm='ortho')
        ft = torch.fft.rfft2(tg, norm='ortho')
        return F.l1_loss(fp.abs(), ft.abs())


def compute_mae(pred, target):
    """Official leaderboard metric: MAE on [0, 255] uint8 scale."""
    return F.l1_loss(pred * 255.0, target * 255.0).item()


def compute_psnr(pred, target):
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=[1, 2, 3])
    return (10 * torch.log10(1.0 / (mse + 1e-10))).mean().item()


crit_pixel = CharbonnierLoss(eps=1e-3).to(device)
crit_freq  = FrequencyLoss().to(device)
print('Loss functions ready')

Loss functions ready


## Cell 6 — Training Loop

In [ ]:
optimizer = optim.AdamW(generator.parameters(), lr=LR, weight_decay=1e-4)
# Cosine Annealing with Warm Restarts — helps escape local minima
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)
CKPT_PATH = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_leaf_sr_generator.pth')

best_mae = float('inf')
history  = {'train_loss': [], 'val_mae': [], 'val_psnr': []}

print(f'=== Training for {EPOCHS} epochs on {device} ===')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    generator.train()
    total_loss = 0.0
    for lr_b, hr_b, _ in train_loader:
        lr_b, hr_b = lr_b.to(device), hr_b.to(device)
        optimizer.zero_grad()
        sr = generator(lr_b)
        loss = crit_pixel(sr, hr_b) + FREQ_WEIGHT * crit_freq(sr, hr_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * lr_b.size(0)
    scheduler.step()
    avg_loss = (total_loss / len(train_ds)) * 255.0

    # ── Validate ──
    generator.eval()
    val_maes, val_psnrs = [], []
    with torch.no_grad():
        for lr_b, hr_b, _ in val_loader:
            lr_b, hr_b = lr_b.to(device), hr_b.to(device)
            pred = generator(lr_b)
            val_maes.append(compute_mae(pred, hr_b))
            val_psnrs.append(compute_psnr(pred, hr_b))
    vmae, vpsnr = np.mean(val_maes), np.mean(val_psnrs)

    history['train_loss'].append(avg_loss)
    history['val_mae'].append(vmae)
    history['val_psnr'].append(vpsnr)

    flag = ' ← 🏆 BEATS BILATERAL!' if vmae < 17.487 else ''
    print(f'Epoch [{epoch:03d}/{EPOCHS}] ({time.time()-t0:.1f}s) | '
          f'Train Loss: {avg_loss:.4f} | Val MAE: {vmae:.4f} | PSNR: {vpsnr:.2f} dB{flag}')

    if vmae < best_mae:
        best_mae = vmae
        torch.save(generator.state_dict(), CKPT_PATH)
        print(f'  --> Checkpoint saved (Val MAE: {best_mae:.4f})')

print(f'\nTraining complete! Best Val MAE: {best_mae:.4f}')

## Cell 7 — Training Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history['val_mae'], label='Val MAE', color='royalblue')
ax1.axhline(17.487, color='orange', linestyle='--', label='Bilateral baseline (17.487)')
ax1.axhline(17.667, color='red',    linestyle='--', label='Rank-1 on LB (17.667)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MAE'); ax1.set_title('Validation MAE')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['val_psnr'], label='Val PSNR', color='seagreen')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('PSNR (dB)'); ax2.set_title('Validation PSNR')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve.png'), dpi=150)
plt.show()

## Cell 8 — Inference with 8× TTA + Submission

In [ ]:
# Load best checkpoint
generator.load_state_dict(torch.load(CKPT_PATH, map_location=device))
generator.eval()
print(f'Loaded best checkpoint (Val MAE: {best_mae:.4f})')


def predict_tta(model, x):
    """8-fold TTA: 4 rotations × 2 (original + horizontal flip)."""
    preds = []
    for k in [0, 1, 2, 3]:
        xr = torch.rot90(x, k=k, dims=[2, 3])
        preds.append(torch.rot90(model(xr), k=(4 - k) % 4, dims=[2, 3]))
        xf = torch.flip(xr, dims=[3])
        preds.append(torch.rot90(torch.flip(model(xf), dims=[3]), k=(4 - k) % 4, dims=[2, 3]))
    return torch.stack(preds, 0).mean(0)


pred_dir = '/kaggle/working/predictions'
os.makedirs(pred_dir, exist_ok=True)
preds_dict = {}

with torch.no_grad():
    for lr_b, fnames in test_loader:
        sr = predict_tta(generator, lr_b.to(device))
        sr_np = (sr.permute(0, 2, 3, 1).cpu().numpy() * 255.0).round().clip(0, 255).astype(np.uint8)
        for i, fn in enumerate(fnames):
            Image.fromarray(sr_np[i]).save(os.path.join(pred_dir, fn))
            preds_dict[fn] = sr_np[i].reshape(-1)

print(f'Generated {len(preds_dict)} predictions')

# Build submission CSV
test_df = pd.read_csv(TEST_CSV_PATH)
sub = test_df[['id']].copy()
sub['pixel_value'] = [preds_dict[img][idx]
                      for img, idx in zip(test_df['image'], test_df['pixel_index'])]
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Wrote {len(sub):,} rows → /kaggle/working/submission.csv')
print(sub.head())

## Cell 9 — Validation Visualization

In [ ]:
generator.eval()
lr_batch, hr_batch, fnames = next(iter(val_loader))
lr_batch, hr_batch = lr_batch[:4].to(device), hr_batch[:4].to(device)

with torch.no_grad():
    pred_batch = generator(lr_batch)
    bilin_batch = F.interpolate(lr_batch, scale_factor=4, mode='bilinear', align_corners=False)

to_np = lambda t: (t.permute(0,2,3,1).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
lr_np, bil_np, pred_np, hr_np = to_np(lr_batch), to_np(bilin_batch), to_np(pred_batch), to_np(hr_batch)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(4):
    b_mae = compute_mae(bilin_batch[i:i+1], hr_batch[i:i+1])
    m_mae = compute_mae(pred_batch[i:i+1],  hr_batch[i:i+1])
    axes[i,0].imshow(lr_np[i]);   axes[i,0].set_title(f'Input 32×32\n{fnames[i][:20]}')
    axes[i,1].imshow(bil_np[i]);  axes[i,1].set_title(f'Bilinear\nMAE:{b_mae:.2f}')
    axes[i,2].imshow(pred_np[i]); axes[i,2].set_title(f'Model\nMAE:{m_mae:.2f}')
    axes[i,3].imshow(hr_np[i]);   axes[i,3].set_title('Ground Truth 128×128')
    for ax in axes[i]: ax.axis('off')

plt.suptitle('Validation Samples: LR → Bilinear → Model → GT', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'validation_grid.png'), dpi=150)
plt.show()